In [1]:
# ============================================================
# TEST HYBRID MODEL: Xception-DenseNet121
# ============================================================
# Based on quick test results:
# - DenseNet121 = 83.74% (best single model)
# - Now test hybrid to see if it's better
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import Xception, DenseNet121
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, balanced_accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# Pre-download weights
print("📥 Loading weights...")
_ = Xception(weights='imagenet', include_top=False)
_ = DenseNet121(weights='imagenet', include_top=False)
print("✅ Weights cached!")
tf.keras.backend.clear_session()
gc.collect()

# ============================================================
# CONFIG
# ============================================================

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CONFIG = {
    'data_dir': '/content/drive/MyDrive/BreakHis_Split',
    'output_dir': '/content/drive/MyDrive/BreakHis_PLOS_ONE_v3',
    'image_size': (224, 224),
    'batch_size': 32,
    'epochs': 30,
    'learning_rate': 0.001,
    'n_folds': 5,
    'early_stopping_patience': 5,
    'reduce_lr_patience': 2,
}

# ============================================================
# HYBRID MODEL: Xception + DenseNet121
# ============================================================

data_augmentation = keras.Sequential([
    layers.RandomRotation(0.15),
    layers.RandomFlip("horizontal"),
    layers.RandomFlip("vertical"),
    layers.RandomZoom(0.15),
], name='augmentation')


def build_xception_densenet121(input_shape=(224, 224, 3)):
    """
    PROPOSED HYBRID MODEL
    Combines:
    - Xception: Edge detection (depthwise separable convolutions)
    - DenseNet121: Texture extraction (dense connections for LBP features)
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Base models - FROZEN
    xception = Xception(weights='imagenet', include_top=False, pooling='avg')
    densenet = DenseNet121(weights='imagenet', include_top=False, pooling='avg')

    xception.trainable = False
    densenet.trainable = False

    # Extract and concatenate features
    x1 = xception(x)  # 2048 features
    x2 = densenet(x)  # 1024 features
    concat = layers.Concatenate()([x1, x2])  # 3072 features

    # Classification head
    x = layers.BatchNormalization()(concat)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='Xception_DenseNet121')


# ============================================================
# DATA LOADING (Same as before)
# ============================================================

def get_patient_data(data_dir):
    """Get patient info without loading images."""
    print("\n📂 Scanning patient data...")
    images = []
    cr_level = 'CR1_original'

    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(data_dir, cr_level, split)

        for class_idx, class_name in enumerate(['benign', 'malignant']):
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(class_dir):
                continue

            for mag in os.listdir(class_dir):
                mag_dir = os.path.join(class_dir, mag)
                if not os.path.isdir(mag_dir):
                    continue

                for filename in os.listdir(mag_dir):
                    if filename.endswith('.png'):
                        parts = filename.replace('.png', '').split('-')
                        patient_id = f"{parts[1]}-{parts[2]}" if len(parts) >= 3 else 'unknown'
                        full_path = os.path.join(mag_dir, filename)

                        images.append({
                            'filepath': full_path,
                            'filename': filename,
                            'class_name': class_name,
                            'class_idx': class_idx,
                            'patient_id': patient_id,
                        })

    df = pd.DataFrame(images)
    print(f"  Total: {len(df):,} images from {df['patient_id'].nunique()} patients")
    return df


def create_dataset(filepaths, labels, batch_size, shuffle=True):
    """Create tf.data.Dataset from file paths."""

    def load_image(filepath, label):
        img = tf.io.read_file(filepath)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, CONFIG['image_size'])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(filepaths), seed=SEED)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


def compute_metrics(y_true, y_pred_proba, threshold=0.5):
    """Compute all metrics."""
    y_pred = (y_pred_proba >= threshold).astype(int)
    if len(np.unique(y_true)) < 2:
        return None

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'auc_roc': roc_auc_score(y_true, y_pred_proba),
    }


# ============================================================
# 5-FOLD CV FOR HYBRID MODEL
# ============================================================

def run_hybrid_test(image_df):
    """Run 5-fold CV for hybrid model."""

    print("\n" + "="*60)
    print("🔧 Testing Xception-DenseNet121 HYBRID")
    print("="*60)

    patient_classes = image_df.groupby('patient_id')['class_idx'].first().reset_index()
    patients = patient_classes['patient_id'].values
    labels = patient_classes['class_idx'].values

    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=SEED)

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(patients, labels)):
        print(f"\n┌─ FOLD {fold + 1}/{CONFIG['n_folds']} {'─'*45}┐")

        train_patients = set(patients[train_idx])
        test_patients = set(patients[test_idx])

        # Split train into train/val
        train_patient_list = list(train_patients)
        np.random.seed(SEED + fold)
        np.random.shuffle(train_patient_list)
        n_train = int(len(train_patient_list) * 0.85)
        val_patients = set(train_patient_list[n_train:])
        train_patients = set(train_patient_list[:n_train])

        # Get file paths
        train_df = image_df[image_df['patient_id'].isin(train_patients)]
        val_df = image_df[image_df['patient_id'].isin(val_patients)]
        test_df = image_df[image_df['patient_id'].isin(test_patients)]

        print(f"│ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

        # Class weights
        class_weights = compute_class_weight('balanced',
                                             classes=np.unique(train_df['class_idx']),
                                             y=train_df['class_idx'].values)
        class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
        print(f"│ Class weights: B={class_weights[0]:.2f}, M={class_weights[1]:.2f}")

        # Create datasets
        train_ds = create_dataset(train_df['filepath'].values, train_df['class_idx'].values,
                                  CONFIG['batch_size'], shuffle=True)
        val_ds = create_dataset(val_df['filepath'].values, val_df['class_idx'].values,
                                CONFIG['batch_size'], shuffle=False)

        # Build model
        print(f"│ Building HYBRID model...")
        model = build_xception_densenet121()

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=CONFIG['early_stopping_patience'],
                         restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                             patience=CONFIG['reduce_lr_patience'], verbose=1)
        ]

        print(f"└─ Training {'─'*47}┘")

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=CONFIG['epochs'],
            class_weight=class_weight_dict,
            callbacks=callbacks,
            verbose=1
        )

        print(f"\n✅ Trained {len(history.history['loss'])} epochs")

        # Evaluate
        print(f"📊 Evaluating...")
        test_ds = create_dataset(test_df['filepath'].values, test_df['class_idx'].values,
                                 CONFIG['batch_size'], shuffle=False)

        y_pred_proba = model.predict(test_ds, verbose=0).flatten()
        y_true = test_df['class_idx'].values

        metrics = compute_metrics(y_true, y_pred_proba)

        if metrics:
            metrics['fold'] = fold + 1
            metrics['model'] = 'Xception_DenseNet121'
            fold_results.append(metrics)

            print(f"\n┌─ FOLD {fold + 1} RESULTS {'─'*40}┐")
            print(f"│ Accuracy:  {metrics['accuracy']:.4f}")
            print(f"│ F1-Score:  {metrics['f1_score']:.4f}")
            print(f"│ AUC-ROC:   {metrics['auc_roc']:.4f}")
            print(f"│ MCC:       {metrics['mcc']:.4f}")
            print(f"└{'─'*55}┘")

        del model, train_ds, val_ds, test_ds
        gc.collect()
        tf.keras.backend.clear_session()

    # Summary
    if fold_results:
        accs = [r['accuracy'] for r in fold_results]
        f1s = [r['f1_score'] for r in fold_results]
        aucs = [r['auc_roc'] for r in fold_results]

        print(f"\n╔═ Xception-DenseNet121 SUMMARY {'═'*25}╗")
        print(f"║ Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
        print(f"║ F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
        print(f"║ AUC-ROC:  {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
        print(f"╚{'═'*55}╝")

    return fold_results


# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':

    print("\n" + "="*60)
    print("🚀 TESTING HYBRID MODEL: Xception-DenseNet121")
    print(f"   Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)

    # Load data
    image_df = get_patient_data(CONFIG['data_dir'])

    # Run hybrid test
    hybrid_results = run_hybrid_test(image_df)

    # Save results
    results_df = pd.DataFrame(hybrid_results)
    results_df.to_csv(os.path.join(CONFIG['output_dir'], 'hybrid_test_results.csv'), index=False)

    # Compare with DenseNet121
    print("\n" + "="*60)
    print("📊 COMPARISON: Hybrid vs DenseNet121")
    print("="*60)

    hybrid_acc = np.mean([r['accuracy'] for r in hybrid_results])
    densenet_acc = 0.8374  # From previous test

    print(f"\nDenseNet121:           {densenet_acc:.4f} (83.74%)")
    print(f"Xception-DenseNet121:  {hybrid_acc:.4f} ({hybrid_acc*100:.2f}%)")

    if hybrid_acc > densenet_acc:
        improvement = (hybrid_acc - densenet_acc) * 100
        print(f"\n✅ HYBRID IS BETTER by +{improvement:.2f}%!")
        print("   → Use hybrid as main proposed model")
    else:
        diff = (densenet_acc - hybrid_acc) * 100
        print(f"\n⚠️ DenseNet121 is better by +{diff:.2f}%")
        print("   → Consider using DenseNet121 alone")

    print(f"\n✅ Complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Mounted at /content/drive
📥 Loading weights...
83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
✅ Weights cached!

🚀 TESTING HYBRID MODEL: Xception-DenseNet121
   Time: 2026-02-26 17:51:48

📂 Scanning patient data...
  Total: 7,910 images from 81 patients

🔧 Testing Xception-DenseNet121 HYBRID

┌─ FOLD 1/5 ─────────────────────────────────────────────┐
│ Train: 5583 | Val: 871 | Test: 1456
│ Class weights: B=1.42, M=0.77
│ Building HYBRID model...
└─ Training ───────────────────────────────────────────────┘
Epoch 1/30
175/175 ━━━━━━━━━━━━━━━━━━━━ 424s 2s/step - accuracy: 0.7077 - loss: 0.7052 - val_accuracy: 0.8576 - val_loss: 0.4255 - learning_rate: 0.0010
Epoch 2/30
175/175 ━━━━━━━━━━━━━━━━━━━━ 29s 165ms/step - accuracy: 0.8100 - loss: 0.4246 - val_accuracy: 0.8657 - val_loss: 0.3508 - learning_rate: 0.0010
Epoch 3/30
175/175 ━━━━━━━━━━━━━━━━━━━━ 29s 164ms/step - accuracy: 0.8378 - loss: 0.3764 - val_accuracy: 0.8450 - val_loss: 0.

In [3]:
# ============================================================
# TEST NEW HYBRID ARCHITECTURES
# ============================================================
# 1. DenseNet169 alone (deeper texture)
# 2. Attention-Fusion Hybrid (learns optimal weighting)
# 3. Xception-DenseNet169 (if time permits)
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    Xception, DenseNet121, DenseNet169, InceptionV3
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, balanced_accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# Pre-download weights
print("="*60)
print("📥 PRE-DOWNLOADING WEIGHTS")
print("="*60)

for name, fn in [('Xception', Xception), ('DenseNet121', DenseNet121),
                  ('DenseNet169', DenseNet169), ('InceptionV3', InceptionV3)]:
    print(f"  {name}...", end=" ")
    _ = fn(weights='imagenet', include_top=False)
    print("✅")

print("\n✅ All weights cached!")
tf.keras.backend.clear_session()
gc.collect()

# ============================================================
# CONFIG
# ============================================================

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CONFIG = {
    'data_dir': '/content/drive/MyDrive/BreakHis_Split',
    'output_dir': '/content/drive/MyDrive/BreakHis_PLOS_ONE_v4',
    'image_size': (224, 224),
    'batch_size': 32,
    'epochs': 30,
    'learning_rate': 0.001,
    'n_folds': 5,
    'early_stopping_patience': 5,
    'reduce_lr_patience': 2,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Data augmentation
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.15),
    layers.RandomFlip("horizontal"),
    layers.RandomFlip("vertical"),
    layers.RandomZoom(0.15),
], name='augmentation')


# ============================================================
# MODEL 1: DenseNet169 (Deeper Texture Extraction)
# ============================================================

def build_densenet169(input_shape=(224, 224, 3)):
    """
    DenseNet169 - Deeper version for enhanced texture capture.
    More dense blocks = better LBP feature extraction.
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    base = DenseNet169(weights='imagenet', include_top=False, pooling='avg')
    base.trainable = False

    x = base(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='DenseNet169')


# ============================================================
# MODEL 2: ATTENTION-FUSION HYBRID (Novel!) ⭐
# ============================================================

class ChannelAttention(layers.Layer):
    """Channel attention module to weight feature importance."""
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channel = input_shape[-1]
        self.shared_dense1 = layers.Dense(channel // self.ratio, activation='relu')
        self.shared_dense2 = layers.Dense(channel, activation='sigmoid')
        super().build(input_shape)

    def call(self, x):
        # Global average pooling
        avg_pool = tf.reduce_mean(x, axis=1, keepdims=True)
        # Global max pooling
        max_pool = tf.reduce_max(x, axis=1, keepdims=True)

        # Shared MLP
        avg_out = self.shared_dense2(self.shared_dense1(avg_pool))
        max_out = self.shared_dense2(self.shared_dense1(max_pool))

        # Combine
        attention = avg_out + max_out
        return x * attention


def build_attention_fusion_hybrid(input_shape=(224, 224, 3)):
    """
    NOVEL ATTENTION-FUSION HYBRID

    Key innovations:
    1. Uses attention mechanism to learn optimal feature weighting
    2. Combines Xception (edges) + DenseNet121 (texture) with learned fusion
    3. Attention learns which features matter for each sample

    This is scientifically justified because:
    - Different regions may need different features (edges vs texture)
    - Attention can emphasize LBP-like features when they're discriminative
    - More flexible than simple concatenation
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Branch 1: Xception (edge/structure features)
    xception = Xception(weights='imagenet', include_top=False, pooling='avg')
    xception.trainable = False
    x1 = xception(x)  # (batch, 2048)

    # Branch 2: DenseNet121 (texture features)
    densenet = DenseNet121(weights='imagenet', include_top=False, pooling='avg')
    densenet.trainable = False
    x2 = densenet(x)  # (batch, 1024)

    # Project to same dimension
    x1 = layers.Dense(512, activation='relu', name='xception_proj')(x1)
    x2 = layers.Dense(512, activation='relu', name='densenet_proj')(x2)

    # Stack features for attention
    x1_expand = layers.Reshape((1, 512))(x1)
    x2_expand = layers.Reshape((1, 512))(x2)
    stacked = layers.Concatenate(axis=1)([x1_expand, x2_expand])  # (batch, 2, 512)

    # Apply channel attention to learn feature importance
    attended = ChannelAttention(ratio=4)(stacked)

    # Flatten attended features
    fused = layers.Flatten()(attended)  # (batch, 1024)

    # Classification head
    x = layers.BatchNormalization()(fused)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='Attention_Fusion_Hybrid')


# ============================================================
# MODEL 3: Xception-DenseNet169 (Deeper Hybrid)
# ============================================================

def build_xception_densenet169(input_shape=(224, 224, 3)):
    """
    Xception + DenseNet169 hybrid.
    Combines edge detection with deeper texture extraction.
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Xception branch
    xception = Xception(weights='imagenet', include_top=False, pooling='avg')
    xception.trainable = False
    x1 = xception(x)  # 2048 features

    # DenseNet169 branch
    densenet169 = DenseNet169(weights='imagenet', include_top=False, pooling='avg')
    densenet169.trainable = False
    x2 = densenet169(x)  # 1664 features

    # Concatenate
    concat = layers.Concatenate()([x1, x2])  # 3712 features

    # Classification head
    x = layers.BatchNormalization()(concat)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='Xception_DenseNet169')


# ============================================================
# MODEL 4: Multi-Scale DenseNet (Pyramid Features)
# ============================================================

def build_multiscale_densenet(input_shape=(224, 224, 3)):
    """
    Multi-scale DenseNet with feature pyramid.
    Extracts features at multiple scales for varying cell sizes.
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Base DenseNet121 with multiple outputs
    base = DenseNet121(weights='imagenet', include_top=False)
    base.trainable = False

    # Get intermediate features at different scales
    # DenseNet121 has 4 dense blocks
    layer_names = [
        'pool2_pool',      # After block 1: 56x56
        'pool3_pool',      # After block 2: 28x28
        'pool4_pool',      # After block 3: 14x14
        'relu',            # Final: 7x7
    ]

    outputs_list = []
    for name in layer_names:
        try:
            layer_output = base.get_layer(name).output
            pooled = layers.GlobalAveragePooling2D()(layer_output)
            outputs_list.append(pooled)
        except:
            pass

    # Build feature extractor
    feature_extractor = Model(inputs=base.input, outputs=outputs_list)

    # Extract multi-scale features
    features = feature_extractor(x)

    # Concatenate all scales
    if isinstance(features, list):
        concat = layers.Concatenate()(features)
    else:
        concat = features

    # Classification head
    x = layers.BatchNormalization()(concat)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='MultiScale_DenseNet')


# ============================================================
# ALL MODELS TO TEST
# ============================================================

MODELS = {
    'DenseNet169': build_densenet169,
    'Attention_Fusion_Hybrid': build_attention_fusion_hybrid,
    'Xception_DenseNet169': build_xception_densenet169,
}

print(f"\n📋 Models to test: {list(MODELS.keys())}")


# ============================================================
# DATA LOADING (Same as before)
# ============================================================

def get_patient_data(data_dir):
    """Get patient info without loading images."""
    print("\n📂 Scanning patient data...")
    images = []
    cr_level = 'CR1_original'

    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(data_dir, cr_level, split)

        for class_idx, class_name in enumerate(['benign', 'malignant']):
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(class_dir):
                continue

            for mag in os.listdir(class_dir):
                mag_dir = os.path.join(class_dir, mag)
                if not os.path.isdir(mag_dir):
                    continue

                for filename in os.listdir(mag_dir):
                    if filename.endswith('.png'):
                        parts = filename.replace('.png', '').split('-')
                        patient_id = f"{parts[1]}-{parts[2]}" if len(parts) >= 3 else 'unknown'
                        full_path = os.path.join(mag_dir, filename)

                        images.append({
                            'filepath': full_path,
                            'filename': filename,
                            'class_name': class_name,
                            'class_idx': class_idx,
                            'patient_id': patient_id,
                        })

    df = pd.DataFrame(images)
    print(f"  Total: {len(df):,} images from {df['patient_id'].nunique()} patients")
    return df


def create_dataset(filepaths, labels, batch_size, shuffle=True):
    """Create tf.data.Dataset from file paths."""

    def load_image(filepath, label):
        img = tf.io.read_file(filepath)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, CONFIG['image_size'])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(filepaths), seed=SEED)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


def compute_metrics(y_true, y_pred_proba, threshold=0.5):
    """Compute all metrics."""
    y_pred = (y_pred_proba >= threshold).astype(int)
    if len(np.unique(y_true)) < 2:
        return None

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'auc_roc': roc_auc_score(y_true, y_pred_proba),
    }


# ============================================================
# 5-FOLD CV
# ============================================================

def run_5fold_cv(model_name, build_fn, image_df):
    """Run 5-fold CV."""

    print(f"\n{'='*60}")
    print(f"🔧 {model_name}")
    print(f"{'='*60}")

    patient_classes = image_df.groupby('patient_id')['class_idx'].first().reset_index()
    patients = patient_classes['patient_id'].values
    labels = patient_classes['class_idx'].values

    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=SEED)

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(patients, labels)):
        print(f"\n┌─ FOLD {fold + 1}/{CONFIG['n_folds']} {'─'*45}┐")

        train_patients = set(patients[train_idx])
        test_patients = set(patients[test_idx])

        # Split train into train/val
        train_patient_list = list(train_patients)
        np.random.seed(SEED + fold)
        np.random.shuffle(train_patient_list)
        n_train = int(len(train_patient_list) * 0.85)
        val_patients = set(train_patient_list[n_train:])
        train_patients = set(train_patient_list[:n_train])

        # Get file paths
        train_df = image_df[image_df['patient_id'].isin(train_patients)]
        val_df = image_df[image_df['patient_id'].isin(val_patients)]
        test_df = image_df[image_df['patient_id'].isin(test_patients)]

        print(f"│ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

        # Class weights
        class_weights = compute_class_weight('balanced',
                                             classes=np.unique(train_df['class_idx']),
                                             y=train_df['class_idx'].values)
        class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

        # Create datasets
        train_ds = create_dataset(train_df['filepath'].values, train_df['class_idx'].values,
                                  CONFIG['batch_size'], shuffle=True)
        val_ds = create_dataset(val_df['filepath'].values, val_df['class_idx'].values,
                                CONFIG['batch_size'], shuffle=False)

        # Build model
        print(f"│ Building model...")
        model = build_fn()

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=CONFIG['early_stopping_patience'],
                         restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                             patience=CONFIG['reduce_lr_patience'], verbose=1)
        ]

        print(f"└─ Training {'─'*47}┘")

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=CONFIG['epochs'],
            class_weight=class_weight_dict,
            callbacks=callbacks,
            verbose=1
        )

        print(f"\n✅ Trained {len(history.history['loss'])} epochs")

        # Evaluate
        print(f"📊 Evaluating...")
        test_ds = create_dataset(test_df['filepath'].values, test_df['class_idx'].values,
                                 CONFIG['batch_size'], shuffle=False)

        y_pred_proba = model.predict(test_ds, verbose=0).flatten()
        y_true = test_df['class_idx'].values

        metrics = compute_metrics(y_true, y_pred_proba)

        if metrics:
            metrics['fold'] = fold + 1
            metrics['model'] = model_name
            fold_results.append(metrics)

            print(f"\n┌─ FOLD {fold + 1} RESULTS {'─'*40}┐")
            print(f"│ Accuracy:  {metrics['accuracy']:.4f}")
            print(f"│ F1-Score:  {metrics['f1_score']:.4f}")
            print(f"│ AUC-ROC:   {metrics['auc_roc']:.4f}")
            print(f"│ MCC:       {metrics['mcc']:.4f}")
            print(f"└{'─'*55}┘")

        del model, train_ds, val_ds, test_ds
        gc.collect()
        tf.keras.backend.clear_session()

    # Summary
    if fold_results:
        accs = [r['accuracy'] for r in fold_results]
        f1s = [r['f1_score'] for r in fold_results]
        aucs = [r['auc_roc'] for r in fold_results]

        print(f"\n╔═ {model_name} SUMMARY {'═'*35}╗")
        print(f"║ Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
        print(f"║ F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
        print(f"║ AUC-ROC:  {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
        print(f"╚{'═'*55}╝")

    return fold_results


# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':

    print("\n" + "="*60)
    print("🚀 TESTING NEW HYBRID ARCHITECTURES")
    print(f"   Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)

    # Load data
    image_df = get_patient_data(CONFIG['data_dir'])

    # Test all models
    all_results = []

    for model_name, build_fn in MODELS.items():
        print(f"\n\n{'#'*60}")
        print(f"# {model_name}")
        print(f"{'#'*60}")

        results = run_5fold_cv(model_name, build_fn, image_df)
        all_results.extend(results)

        # Save after each model
        results_df = pd.DataFrame(all_results)
        results_df.to_csv(os.path.join(CONFIG['output_dir'], 'hybrid_test_v2.csv'), index=False)

    # ==================== FINAL COMPARISON ====================

    print("\n\n" + "="*70)
    print("📊 FINAL COMPARISON - ALL MODELS")
    print("="*70)

    # Previous results
    previous_results = {
        'DenseNet121': {'accuracy': 0.8374, 'f1': 0.8817, 'auc': 0.8873},
        'Xception_DenseNet121': {'accuracy': 0.8346, 'f1': 0.8816, 'auc': 0.8669},
    }

    results_df = pd.DataFrame(all_results)

    print(f"\n{'Model':<30} {'Accuracy':<20} {'F1-Score':<15} {'AUC-ROC':<15}")
    print("-"*70)

    # Print previous results
    for model, metrics in previous_results.items():
        print(f"{model:<30} {metrics['accuracy']:.4f}            {metrics['f1']:.4f}         {metrics['auc']:.4f}")

    print("-"*70)

    # Print new results
    for model in MODELS.keys():
        model_data = results_df[results_df['model'] == model]
        if len(model_data) > 0:
            acc = f"{model_data['accuracy'].mean():.4f} ± {model_data['accuracy'].std():.4f}"
            f1 = f"{model_data['f1_score'].mean():.4f}"
            auc = f"{model_data['auc_roc'].mean():.4f}"
            print(f"{model:<30} {acc:<20} {f1:<15} {auc:<15}")

    # Best model
    best_acc = 0
    best_model = ""

    for model, metrics in previous_results.items():
        if metrics['accuracy'] > best_acc:
            best_acc = metrics['accuracy']
            best_model = model

    for model in MODELS.keys():
        model_data = results_df[results_df['model'] == model]
        if len(model_data) > 0:
            acc = model_data['accuracy'].mean()
            if acc > best_acc:
                best_acc = acc
                best_model = model

    print("\n" + "="*70)
    print(f"🏆 BEST MODEL: {best_model} ({best_acc:.4f})")
    print("="*70)

    print(f"\n✅ Complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"📁 Results saved to: {CONFIG['output_dir']}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 PRE-DOWNLOADING WEIGHTS
  Xception... ✅
  DenseNet121... ✅
  DenseNet169... Downloading data from https://storage.googleapis.com/tensorflow/keras-applications/densenet/densenet169_weights_tf_dim_ordering_tf_kernels_notop.h5
51877672/51877672 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
✅
  InceptionV3... Downloading data from https://storage.googleapis.com/tensorflow/keras-applications/inception_v3/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
✅

✅ All weights cached!

📋 Models to test: ['DenseNet169', 'Attention_Fusion_Hybrid', 'Xception_DenseNet169']

🚀 TESTING NEW HYBRID ARCHITECTURES
   Time: 2026-02-26 18:41:17

📂 Scanning patient data...
  Total: 7,910 images from 81 patients


############################################################
# DenseNet169
###################################################

In [4]:
# ============================================================
# NEW TEXTURE-FOCUSED HYBRID ARCHITECTURES
# ============================================================
#
# Key insight: Replace Xception with texture-focused alternatives
#
# 1. DenseNet121 + DenseNet169 Attention Fusion
# 2. InceptionV3 + DenseNet169 Attention Fusion
# 3. SE-DenseNet169 (Self-attention on texture)
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    DenseNet121, DenseNet169, InceptionV3
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, balanced_accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# Pre-download weights
print("="*60)
print("📥 PRE-DOWNLOADING WEIGHTS")
print("="*60)

for name, fn in [('DenseNet121', DenseNet121), ('DenseNet169', DenseNet169),
                  ('InceptionV3', InceptionV3)]:
    print(f"  {name}...", end=" ")
    _ = fn(weights='imagenet', include_top=False)
    print("✅")

print("\n✅ All weights cached!")
tf.keras.backend.clear_session()
gc.collect()

# ============================================================
# CONFIG
# ============================================================

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CONFIG = {
    'data_dir': '/content/drive/MyDrive/BreakHis_Split',
    'output_dir': '/content/drive/MyDrive/BreakHis_PLOS_ONE_v5',
    'image_size': (224, 224),
    'batch_size': 32,
    'epochs': 30,
    'learning_rate': 0.001,
    'n_folds': 5,
    'early_stopping_patience': 5,
    'reduce_lr_patience': 2,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Data augmentation
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.15),
    layers.RandomFlip("horizontal"),
    layers.RandomFlip("vertical"),
    layers.RandomZoom(0.15),
], name='augmentation')


# ============================================================
# ATTENTION MODULES
# ============================================================

class ChannelAttention(layers.Layer):
    """Squeeze-and-Excitation style channel attention."""
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channel = input_shape[-1]
        self.shared_dense1 = layers.Dense(channel // self.ratio, activation='relu')
        self.shared_dense2 = layers.Dense(channel, activation='sigmoid')
        super().build(input_shape)

    def call(self, x):
        # Global average pooling
        avg_pool = tf.reduce_mean(x, axis=1, keepdims=True)
        # Global max pooling
        max_pool = tf.reduce_max(x, axis=1, keepdims=True)

        # Shared MLP
        avg_out = self.shared_dense2(self.shared_dense1(avg_pool))
        max_out = self.shared_dense2(self.shared_dense1(max_pool))

        # Combine
        attention = avg_out + max_out
        return x * attention


class SEBlock(layers.Layer):
    """Squeeze-and-Excitation Block for feature recalibration."""
    def __init__(self, ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channels = input_shape[-1]
        self.gap = layers.GlobalAveragePooling2D()
        self.fc1 = layers.Dense(channels // self.ratio, activation='relu')
        self.fc2 = layers.Dense(channels, activation='sigmoid')
        self.reshape = layers.Reshape((1, 1, channels))
        super().build(input_shape)

    def call(self, x):
        # Squeeze
        se = self.gap(x)
        # Excitation
        se = self.fc1(se)
        se = self.fc2(se)
        se = self.reshape(se)
        # Scale
        return x * se


# ============================================================
# MODEL 1: DenseNet121 + DenseNet169 ATTENTION FUSION ⭐
# ============================================================

def build_dual_densenet_attention(input_shape=(224, 224, 3)):
    """
    DUAL DENSENET ATTENTION FUSION

    Combines two DenseNets at different depths for multi-scale texture:
    - DenseNet121: Captures coarse texture patterns
    - DenseNet169: Captures fine-grained texture details
    - Attention: Learns optimal weighting between scales

    This should work because BOTH branches capture texture (unlike Xception)
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Branch 1: DenseNet121 (coarse texture)
    densenet121 = DenseNet121(weights='imagenet', include_top=False, pooling='avg')
    densenet121.trainable = False
    densenet121._name = 'densenet121'
    x1 = densenet121(x)  # 1024 features

    # Branch 2: DenseNet169 (fine texture)
    densenet169 = DenseNet169(weights='imagenet', include_top=False, pooling='avg')
    densenet169.trainable = False
    densenet169._name = 'densenet169'
    x2 = densenet169(x)  # 1664 features

    # Project to same dimension
    x1 = layers.Dense(512, activation='relu', name='dn121_proj')(x1)
    x1 = layers.BatchNormalization()(x1)

    x2 = layers.Dense(512, activation='relu', name='dn169_proj')(x2)
    x2 = layers.BatchNormalization()(x2)

    # Stack for attention
    x1_exp = layers.Reshape((1, 512))(x1)
    x2_exp = layers.Reshape((1, 512))(x2)
    stacked = layers.Concatenate(axis=1)([x1_exp, x2_exp])  # (batch, 2, 512)

    # Channel attention to learn feature importance
    attended = ChannelAttention(ratio=4)(stacked)

    # Flatten
    fused = layers.Flatten()(attended)  # (batch, 1024)

    # Classification head
    x = layers.BatchNormalization()(fused)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='DualDenseNet_Attention')


# ============================================================
# MODEL 2: InceptionV3 + DenseNet169 ATTENTION FUSION
# ============================================================

def build_inception_densenet_attention(input_shape=(224, 224, 3)):
    """
    INCEPTION + DENSENET ATTENTION FUSION

    Combines:
    - InceptionV3: Multi-scale feature extraction (1x1, 3x3, 5x5 kernels)
    - DenseNet169: Fine-grained texture extraction
    - Attention: Learns optimal combination

    InceptionV3 captures features at multiple scales, complementing
    DenseNet's texture extraction.
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Branch 1: InceptionV3 (multi-scale)
    inception = InceptionV3(weights='imagenet', include_top=False, pooling='avg')
    inception.trainable = False
    x1 = inception(x)  # 2048 features

    # Branch 2: DenseNet169 (texture)
    densenet169 = DenseNet169(weights='imagenet', include_top=False, pooling='avg')
    densenet169.trainable = False
    x2 = densenet169(x)  # 1664 features

    # Project to same dimension
    x1 = layers.Dense(512, activation='relu', name='inception_proj')(x1)
    x1 = layers.BatchNormalization()(x1)

    x2 = layers.Dense(512, activation='relu', name='densenet_proj')(x2)
    x2 = layers.BatchNormalization()(x2)

    # Stack for attention
    x1_exp = layers.Reshape((1, 512))(x1)
    x2_exp = layers.Reshape((1, 512))(x2)
    stacked = layers.Concatenate(axis=1)([x1_exp, x2_exp])

    # Channel attention
    attended = ChannelAttention(ratio=4)(stacked)

    # Flatten
    fused = layers.Flatten()(attended)

    # Classification head
    x = layers.BatchNormalization()(fused)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='Inception_DenseNet_Attention')


# ============================================================
# MODEL 3: SE-DenseNet169 (Self-Attention on Texture)
# ============================================================

def build_se_densenet169(input_shape=(224, 224, 3)):
    """
    SE-DENSENET169 (Squeeze-and-Excitation DenseNet)

    Adds channel attention directly to DenseNet169 output.
    This learns which texture feature channels are most important
    for classification.

    Simpler but effective approach - enhance the best model with attention.
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # DenseNet169 without pooling to keep spatial dimensions
    densenet169 = DenseNet169(weights='imagenet', include_top=False, pooling=None)
    densenet169.trainable = False

    features = densenet169(x)  # (batch, 7, 7, 1664)

    # Apply SE attention
    se_features = SEBlock(ratio=16)(features)

    # Global pooling
    x = layers.GlobalAveragePooling2D()(se_features)

    # Classification head
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='SE_DenseNet169')


# ============================================================
# MODEL 4: CBAM-DenseNet169 (Channel + Spatial Attention)
# ============================================================

class SpatialAttention(layers.Layer):
    """Spatial attention module."""
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same', activation='sigmoid')
        super().build(input_shape)

    def call(self, x):
        # Channel-wise pooling
        avg_pool = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=-1, keepdims=True)
        concat = tf.concat([avg_pool, max_pool], axis=-1)

        # Spatial attention map
        attention = self.conv(concat)
        return x * attention


class CBAMBlock(layers.Layer):
    """Convolutional Block Attention Module."""
    def __init__(self, ratio=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.se = SEBlock(ratio=self.ratio)
        self.sa = SpatialAttention(kernel_size=self.kernel_size)
        super().build(input_shape)

    def call(self, x):
        x = self.se(x)  # Channel attention
        x = self.sa(x)  # Spatial attention
        return x


def build_cbam_densenet169(input_shape=(224, 224, 3)):
    """
    CBAM-DENSENET169 (Channel + Spatial Attention)

    Most sophisticated attention mechanism:
    - Channel attention: WHAT features to focus on
    - Spatial attention: WHERE to focus in the image

    Should be best for histopathology where both
    feature type AND location matter.
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # DenseNet169 without pooling
    densenet169 = DenseNet169(weights='imagenet', include_top=False, pooling=None)
    densenet169.trainable = False

    features = densenet169(x)  # (batch, 7, 7, 1664)

    # Apply CBAM attention
    cbam_features = CBAMBlock(ratio=16, kernel_size=7)(features)

    # Global pooling
    x = layers.GlobalAveragePooling2D()(cbam_features)

    # Classification head
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='CBAM_DenseNet169')


# ============================================================
# ALL MODELS TO TEST
# ============================================================

MODELS = {
    'DualDenseNet_Attention': build_dual_densenet_attention,
    'Inception_DenseNet_Attention': build_inception_densenet_attention,
    'SE_DenseNet169': build_se_densenet169,
    'CBAM_DenseNet169': build_cbam_densenet169,
}

print(f"\n📋 NEW Models to test: {list(MODELS.keys())}")


# ============================================================
# DATA LOADING (Same as before)
# ============================================================

def get_patient_data(data_dir):
    """Get patient info without loading images."""
    print("\n📂 Scanning patient data...")
    images = []
    cr_level = 'CR1_original'

    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(data_dir, cr_level, split)

        for class_idx, class_name in enumerate(['benign', 'malignant']):
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(class_dir):
                continue

            for mag in os.listdir(class_dir):
                mag_dir = os.path.join(class_dir, mag)
                if not os.path.isdir(mag_dir):
                    continue

                for filename in os.listdir(mag_dir):
                    if filename.endswith('.png'):
                        parts = filename.replace('.png', '').split('-')
                        patient_id = f"{parts[1]}-{parts[2]}" if len(parts) >= 3 else 'unknown'
                        full_path = os.path.join(mag_dir, filename)

                        images.append({
                            'filepath': full_path,
                            'filename': filename,
                            'class_name': class_name,
                            'class_idx': class_idx,
                            'patient_id': patient_id,
                        })

    df = pd.DataFrame(images)
    print(f"  Total: {len(df):,} images from {df['patient_id'].nunique()} patients")
    return df


def create_dataset(filepaths, labels, batch_size, shuffle=True):
    """Create tf.data.Dataset from file paths."""

    def load_image(filepath, label):
        img = tf.io.read_file(filepath)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, CONFIG['image_size'])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(filepaths), seed=SEED)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


def compute_metrics(y_true, y_pred_proba, threshold=0.5):
    """Compute all metrics."""
    y_pred = (y_pred_proba >= threshold).astype(int)
    if len(np.unique(y_true)) < 2:
        return None

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'auc_roc': roc_auc_score(y_true, y_pred_proba),
    }


# ============================================================
# 5-FOLD CV
# ============================================================

def run_5fold_cv(model_name, build_fn, image_df):
    """Run 5-fold CV."""

    print(f"\n{'='*60}")
    print(f"🔧 {model_name}")
    print(f"{'='*60}")

    patient_classes = image_df.groupby('patient_id')['class_idx'].first().reset_index()
    patients = patient_classes['patient_id'].values
    labels = patient_classes['class_idx'].values

    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=SEED)

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(patients, labels)):
        print(f"\n┌─ FOLD {fold + 1}/{CONFIG['n_folds']} {'─'*45}┐")

        train_patients = set(patients[train_idx])
        test_patients = set(patients[test_idx])

        # Split train into train/val
        train_patient_list = list(train_patients)
        np.random.seed(SEED + fold)
        np.random.shuffle(train_patient_list)
        n_train = int(len(train_patient_list) * 0.85)
        val_patients = set(train_patient_list[n_train:])
        train_patients = set(train_patient_list[:n_train])

        # Get file paths
        train_df = image_df[image_df['patient_id'].isin(train_patients)]
        val_df = image_df[image_df['patient_id'].isin(val_patients)]
        test_df = image_df[image_df['patient_id'].isin(test_patients)]

        print(f"│ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

        # Class weights
        class_weights = compute_class_weight('balanced',
                                             classes=np.unique(train_df['class_idx']),
                                             y=train_df['class_idx'].values)
        class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

        # Create datasets
        train_ds = create_dataset(train_df['filepath'].values, train_df['class_idx'].values,
                                  CONFIG['batch_size'], shuffle=True)
        val_ds = create_dataset(val_df['filepath'].values, val_df['class_idx'].values,
                                CONFIG['batch_size'], shuffle=False)

        # Build model
        print(f"│ Building {model_name}...")
        model = build_fn()

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=CONFIG['early_stopping_patience'],
                         restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                             patience=CONFIG['reduce_lr_patience'], verbose=1)
        ]

        print(f"└─ Training {'─'*47}┘")

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=CONFIG['epochs'],
            class_weight=class_weight_dict,
            callbacks=callbacks,
            verbose=1
        )

        print(f"\n✅ Trained {len(history.history['loss'])} epochs")

        # Evaluate
        print(f"📊 Evaluating...")
        test_ds = create_dataset(test_df['filepath'].values, test_df['class_idx'].values,
                                 CONFIG['batch_size'], shuffle=False)

        y_pred_proba = model.predict(test_ds, verbose=0).flatten()
        y_true = test_df['class_idx'].values

        metrics = compute_metrics(y_true, y_pred_proba)

        if metrics:
            metrics['fold'] = fold + 1
            metrics['model'] = model_name
            fold_results.append(metrics)

            print(f"\n┌─ FOLD {fold + 1} RESULTS {'─'*40}┐")
            print(f"│ Accuracy:  {metrics['accuracy']:.4f}")
            print(f"│ F1-Score:  {metrics['f1_score']:.4f}")
            print(f"│ AUC-ROC:   {metrics['auc_roc']:.4f}")
            print(f"│ MCC:       {metrics['mcc']:.4f}")
            print(f"└{'─'*55}┘")

        del model, train_ds, val_ds, test_ds
        gc.collect()
        tf.keras.backend.clear_session()

    # Summary
    if fold_results:
        accs = [r['accuracy'] for r in fold_results]
        f1s = [r['f1_score'] for r in fold_results]
        aucs = [r['auc_roc'] for r in fold_results]

        print(f"\n╔═ {model_name} SUMMARY {'═'*35}╗")
        print(f"║ Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
        print(f"║ F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
        print(f"║ AUC-ROC:  {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
        print(f"╚{'═'*55}╝")

    return fold_results


# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':

    print("\n" + "="*60)
    print("🚀 TESTING NEW TEXTURE-FOCUSED HYBRID ARCHITECTURES")
    print(f"   Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    print("\n💡 Key Change: Replaced Xception with texture-focused models")
    print("   - DenseNet121 + DenseNet169 (multi-scale texture)")
    print("   - InceptionV3 + DenseNet169 (multi-scale + texture)")
    print("   - SE-DenseNet169 (self-attention on texture)")
    print("   - CBAM-DenseNet169 (channel + spatial attention)")

    # Load data
    image_df = get_patient_data(CONFIG['data_dir'])

    # Test all models
    all_results = []

    for model_name, build_fn in MODELS.items():
        print(f"\n\n{'#'*60}")
        print(f"# {model_name}")
        print(f"{'#'*60}")

        results = run_5fold_cv(model_name, build_fn, image_df)
        all_results.extend(results)

        # Save after each model
        results_df = pd.DataFrame(all_results)
        results_df.to_csv(os.path.join(CONFIG['output_dir'], 'texture_hybrid_results.csv'), index=False)

    # ==================== FINAL COMPARISON ====================

    print("\n\n" + "="*70)
    print("📊 FINAL COMPARISON - ALL MODELS (Including Previous)")
    print("="*70)

    # Previous best results
    previous_results = {
        'DenseNet169 (baseline)': 0.8477,
        'DenseNet121': 0.8374,
        'Xception_DenseNet121': 0.8346,
        'Xception_DenseNet169': 0.8384,
        'Old_Attention_Fusion': 0.8331,
    }

    results_df = pd.DataFrame(all_results)

    print(f"\n{'Model':<35} {'Accuracy':<20}")
    print("-"*55)

    # Print previous results
    for model, acc in previous_results.items():
        print(f"{model:<35} {acc:.4f}")

    print("-"*55)
    print("NEW TEXTURE-FOCUSED MODELS:")
    print("-"*55)

    # Print new results
    best_acc = 0.8477  # DenseNet169 baseline
    best_model = "DenseNet169 (baseline)"

    for model in MODELS.keys():
        model_data = results_df[results_df['model'] == model]
        if len(model_data) > 0:
            acc_mean = model_data['accuracy'].mean()
            acc_std = model_data['accuracy'].std()
            f1_mean = model_data['f1_score'].mean()
            auc_mean = model_data['auc_roc'].mean()

            marker = "⭐" if acc_mean > best_acc else ""
            print(f"{model:<35} {acc_mean:.4f} ± {acc_std:.4f} {marker}")

            if acc_mean > best_acc:
                best_acc = acc_mean
                best_model = model

    print("\n" + "="*70)
    print(f"🏆 BEST MODEL: {best_model} ({best_acc:.4f})")
    print("="*70)

    if best_acc > 0.8477:
        improvement = (best_acc - 0.8477) * 100
        print(f"\n✅ NEW BEST! +{improvement:.2f}% improvement over DenseNet169!")
        print(f"   → Use {best_model} as main proposed model")
    else:
        print(f"\n⚠️ DenseNet169 baseline still best")
        print(f"   → Use DenseNet169 for compression experiments")

    print(f"\n✅ Complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"📁 Results saved to: {CONFIG['output_dir']}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 PRE-DOWNLOADING WEIGHTS
  DenseNet121... ✅
  DenseNet169... ✅
  InceptionV3... ✅

✅ All weights cached!

📋 NEW Models to test: ['DualDenseNet_Attention', 'Inception_DenseNet_Attention', 'SE_DenseNet169', 'CBAM_DenseNet169']

🚀 TESTING NEW TEXTURE-FOCUSED HYBRID ARCHITECTURES
   Time: 2026-02-26 19:58:54

💡 Key Change: Replaced Xception with texture-focused models
   - DenseNet121 + DenseNet169 (multi-scale texture)
   - InceptionV3 + DenseNet169 (multi-scale + texture)
   - SE-DenseNet169 (self-attention on texture)
   - CBAM-DenseNet169 (channel + spatial attention)

📂 Scanning patient data...
  Total: 7,910 images from 81 patients


############################################################
# DualDenseNet_Attention
############################################################

🔧 DualDenseNet_Attention

┌─ FOLD 1/5 ────────────────────────────────────────